# QuEnAIS Tutorial
## Quantum Embedding for Strongly Correlated Molecules

This notebook walks through the complete pipeline and shows how to customize every step.

### What this package does

```
CIF file
   ↓
Step 0: Classical reference methods (HF, MP2, CCSD, ...)
   ↓
Step 1: Active Space Finder — identifies the most correlated orbitals
   ↓
Step 2: DMET Embedding — builds a compact Hamiltonian (impurity + bath)
   ↓
Step 3: Quantum Solver — SQD / SKQD / SqDRIFT on the embedded Hamiltonian
   ↓
Step 4: Visualization — plots, summary dashboard, CSV export
```

You can run the full pipeline in one command, or run each step independently
and inspect the intermediate results.

## 0. Setup

In [1]:
import os
import numpy as np

# Set your project directory — this is where cif_files/ and results/ will live
PROJECT_DIR = os.path.abspath("../")   # repo root by default
print(f"Project dir: {PROJECT_DIR}")
print(f"CIF files  : {os.path.join(PROJECT_DIR, 'cif_files')}")

Project dir: /home/loharkar/Python-package/quenais
CIF files  : /home/loharkar/Python-package/quenais/cif_files


## 1. Configuration

The `Config` object holds all settings. You never edit a config file —
you create a `Config` object and pass it to each step.

This means you can have multiple configs in one notebook and compare them.

In [2]:
from quenais.config import Config

# Minimal config — sensible defaults for everything
cfg = Config(
    molecule    = "TiO2",
    basis       = "def2-svp",
    project_dir = PROJECT_DIR,
)

print(cfg)
print()
print(f"Results will be saved to: {cfg.results_dir}")
print(f"Plots will be saved to  : {cfg.plots_dir}")

Config(molecule=TiO2, basis=def2-svp, solver=sqd, ansatz=lucj)

Results will be saved to: /home/loharkar/Python-package/quenais/results
Plots will be saved to  : /home/loharkar/Python-package/quenais/results/plots


In [3]:
# All available options shown explicitly
cfg = Config(
    # ── Molecule ──────────────────────────────────────────────────────────
    molecule    = "TiO2",     # must match a .cif file in cif_files/
    basis       = "def2-svp", # any PySCF basis: sto-3g, def2-svp, def2-tzvp, ...
    charge      = 0,          # molecular charge
    spin        = 0,          # 2S: 0=singlet, 2=triplet, 4=quintet
    project_dir = PROJECT_DIR,

    # ── Classical methods (Step 0) ────────────────────────────────────────
    classical_methods = ["HF", "MP2"],
    # Full set: ["HF", "MP2", "CCSD", "CCSD_T", "CASSCF", "NEVPT2"]
    # HF and MP2 take seconds. CCSD takes minutes. CCSD_T and NEVPT2 can take hours.

    # ── Quantum solver (Step 3) ───────────────────────────────────────────
    quantum_solver   = "sqd",   # "sqd" | "skqd" | "sqdrift"
    ansatz           = "lucj",  # "lucj" (recommended) | "su2"
    fermion_to_qubit = "bk",    # "bk" (recommended) | "jw"
    backend          = "mps",   # "mps" (recommended) | "local" | "ibm"
    n_shots          = 8192,
    sqd_iters        = 10,

    # ── LUCJ ansatz parameters ────────────────────────────────────────────
    lucj_num_layers     = 3,    # more layers = more expressive, slower
    lucj_random_seed    = 42,
    lucj_regularization = 1e-2, # stabilises compressed double factorization

    # ── DMET embedding parameters (Step 2) ───────────────────────────────
    bath_tolerance  = 1e-8,  # singular value cutoff for bath selection
    max_embed_orbs  = 24,    # max total orbitals in embedding space
    min_bath_orbs   = 0,     # minimum bath orbitals (0 = allow pure impurity)

    # ── Active space parameters (Step 1) ─────────────────────────────────
    gap_min_norb       = 2,
    gap_max_norb       = 16,
    core_occ_threshold = 1.95,

    # ── Geometry scan (Step 4) ────────────────────────────────────────────
    geometry_scan      = True,
    scan_atom_pair     = (0, 1),          # Ti(0) - O(1) bond
    scan_distances     = np.linspace(0.9, 4.0, 20),  # Angstrom
    scan_method        = "MP2",
    quantum_scan       = True,
    quantum_scan_fast  = True,            # fewer shots for speed
    quantum_scan_shots = 2048,
    quantum_scan_iters = 4,
)

# Validate before running
cfg.validate()
cfg.make_dirs()
cfg.load_geometry()

print(f"Molecule  : {cfg.molecule}")
print(f"Atoms     : {cfg.atom_syms}")
print(f"N atoms   : {cfg.n_atoms}")

Molecule  : TiO2
Atoms     : ['Ti', 'O', 'O']
N atoms   : 3


## 2. Solver Options

### 2a. Quantum Solvers

| Solver | What it does | When to use |
|---|---|---|
| `sqd` | Sample ansatz → iterative subspace diagonalization | Default, fastest |
| `skqd` | Krylov time evolution → cumulative diagonalization | Better for strongly correlated |
| `sqdrift` | Random Trotter circuits → diagonalization | Hardware-friendly |

### 2b. Ansatze

| Ansatz | Particle number | Shots wasted | Speed |
|---|---|---|---|
| `lucj` | ✅ conserved | 0% | Fast |
| `su2` | ❌ not conserved | ~40-60% | Slower |

### 2c. Fermion-to-qubit mappings

| Mapping | Pauli string length | Best for |
|---|---|---|
| `bk` (Bravyi-Kitaev) | O(log N) | Hardware, SKQD |
| `jw` (Jordan-Wigner) | O(N) | Debugging, small systems |

### 2d. Backends

| Backend | Description | When to use |
|---|---|---|
| `mps` | Matrix product state simulator | Default, handles large circuits |
| `local` | Full statevector | Small systems (< 20 qubits) |
| `ibm` | IBM Quantum hardware | Real quantum hardware runs |

## 3. Running the Full Pipeline

In [5]:
from quenais.classical.runner     import main as run_classical
from quenais.active_space.finder  import main as run_asf
from quenais.embedding.hamiltonian import main as run_hamiltonian
from quenais.quantum.solver       import main as run_solver
from quenais.visualization.plots  import main as run_viz

# Run all steps — each step caches its result.
# If a .pkl file exists the step is skipped unless force=True.
run_classical(cfg)
run_asf(cfg)
run_hamiltonian(cfg)
run_solver(cfg)
run_viz(cfg)


[Step 0] Classical Methods — TiO2
  Basis     : def2-svp
  Charge    : 0   Spin (2S): 0
  Methods   : ['HF', 'MP2']
  Electrons : 38   AOs: 59
  Step 1 not found — CASSCF/NEVPT2 will use MP2 natural orbital guess


ModuleNotFoundError: No module named 'pyscf.dmrgscf'

## 4. Running Steps Individually and Inspecting Results

Every step saves a `.pkl` file. You can load and inspect it at any time.

In [ ]:
import pickle

# ── Step 0: Classical ─────────────────────────────────────────────────────────
run_classical(cfg, force=False)  # force=True to recompute

with open(cfg.step0_file, "rb") as f:
    step0 = pickle.load(f)

print("Classical results:")
for method, data in step0["methods"].items():
    e = data.get("energy")
    if e:
        print(f"  {method:10s}: {e:.8f} Ha")

In [ ]:
# ── Step 1: Active Space ──────────────────────────────────────────────────────
run_asf(cfg, force=False)

with open(cfg.step1_file, "rb") as f:
    step1 = pickle.load(f)

print(f"Tier              : {step1['tier']}")
print(f"Active space      : ({step1['nel']}e, {step1['n_active_orbs']}orb)")
print(f"Active MO indices : {step1['mo_list']}")
print(f"Corr. strength    : {step1['corr_strength']:.4f}  (0=weak, 1=max)")
print(f"UHF energy        : {step1['uhf_energy']:.8f} Ha")
print(f"MP2 energy        : {step1['mp2_energy']:.8f} Ha")
print(f"HOMO-LUMO gap     : {step1['indicators']['homo_lumo_gap_eV']:.3f} eV")
print(f"Has TM element    : {step1['indicators']['has_tm']}")

In [ ]:
# ── Step 2: DMET Hamiltonian ──────────────────────────────────────────────────
run_hamiltonian(cfg, force=False)

with open(cfg.step2_file, "rb") as f:
    step2 = pickle.load(f)

print(f"Embedding         : {step2['n_imp']}(imp) + {step2['n_bath']}(bath) = {step2['n_emb']} orbs")
print(f"Qubits needed     : {2 * step2['n_emb']}")
print(f"Electrons         : {step2['n_alpha']}α + {step2['n_beta']}β")
print(f"Core energy       : {step2['ecore']:.6f} Ha")
print(f"sv² coverage      : {step2['sv2_cov']:.4f}")
print(f"h1e shape         : {step2['h1e'].shape}")
print(f"h2e shape         : {step2['h2e'].shape}")

In [ ]:
# ── Step 3: Quantum Solver ────────────────────────────────────────────────────
run_solver(cfg, force=False)

with open(cfg.step3_file, "rb") as f:
    step3 = pickle.load(f)

print(f"Solver    : {step3['solver'].upper()} + {step3['ansatz'].upper()} + {step3['mapping'].upper()}")
print(f"Energy    : {step3['energy']:.8f} Ha")
print(f"vs UHF    : {step3['energy'] - step3['uhf_energy']:+.6f} Ha")
print(f"vs MP2    : {step3['energy'] - step3['mp2_energy']:+.6f} Ha")
print(f"<S²>      : {step3['spin_sq']:.4f}")
print(f"Iterations: {len(step3['iterations'])}")

## 5. Customizing the Workflow

You are not tied to the default pipeline. Every step is a regular Python function.
You can mix and match, skip steps, or replace steps entirely.

In [ ]:
# Example 1: Run only classical methods, skip quantum entirely

cfg_classical_only = Config(
    molecule          = "TiO2",
    basis             = "def2-svp",
    classical_methods = ["HF", "MP2", "CCSD"],
    project_dir       = PROJECT_DIR,
)
cfg_classical_only.validate().make_dirs().load_geometry()

run_classical(cfg_classical_only, force=True)
print("Classical-only run complete.")

In [ ]:
# Example 2: Compare two quantum solvers side by side
# Uses cached Step 1 and Step 2 — only reruns Step 3

import tempfile

for solver, ansatz in [("sqd", "lucj"), ("sqd", "su2"), ("skqd", "su2")]:
    # Each config gets its own results directory to avoid overwriting
    run_dir = tempfile.mkdtemp(prefix=f"quenais_{solver}_{ansatz}_")

    cfg_compare = Config(
        molecule       = "TiO2",
        basis          = "def2-svp",
        quantum_solver = solver,
        ansatz         = ansatz,
        n_shots        = 4096,    # fewer shots for quick comparison
        sqd_iters      = 5,
        project_dir    = run_dir,
    )
    cfg_compare.validate().make_dirs().load_geometry()

    # Copy cached Step 1 and Step 2 from main run to avoid recomputing
    import shutil
    shutil.copy(cfg.step1_file, cfg_compare.step1_file)
    shutil.copy(cfg.step2_file, cfg_compare.step2_file)

    run_solver(cfg_compare, force=True)

    with open(cfg_compare.step3_file, "rb") as f:
        result = pickle.load(f)

    print(f"{solver.upper():8s} + {ansatz.upper():5s}: {result['energy']:.8f} Ha")

In [ ]:
# Example 3: Use the embedding Hamiltonian directly
# You can take h1e and h2e and pass them to any solver you want

with open(cfg.step2_file, "rb") as f:
    step2 = pickle.load(f)

h1e    = step2["h1e"]
h2e    = step2["h2e"]
n_emb  = step2["n_emb"]
n_alpha= step2["n_alpha"]
n_beta = step2["n_beta"]
ecore  = step2["ecore"]

print(f"Embedding Hamiltonian ready:")
print(f"  h1e: {h1e.shape}  (one-body integrals)")
print(f"  h2e: {h2e.shape}  (two-body integrals)")
print(f"  ecore = {ecore:.6f} Ha  (nuclear + core mean-field)")
print()
print("You can now pass h1e, h2e, n_emb, n_alpha, n_beta to any solver.")
print("For example: PySCF FCI, DMRG, or any custom quantum circuit solver.")

In [ ]:
# Example 4: Run FCI on the embedding Hamiltonian as a reference
# (only feasible for small embeddings, n_emb <= 12 or so)

if n_emb <= 12:
    from pyscf import gto, scf, fci as pyscf_fci

    cisolver = pyscf_fci.direct_spin1.FCI()
    cisolver.verbose = 0
    e_fci, _ = cisolver.kernel(
        h1e, h2e, n_emb, (n_alpha, n_beta),
    )
    e_fci_total = float(e_fci) + ecore

    print(f"FCI (embedding)  : {e_fci_total:.8f} Ha")
    print(f"Quantum solver   : {step3['energy']:.8f} Ha")
    print(f"Difference       : {step3['energy'] - e_fci_total:+.6f} Ha")
else:
    print(f"n_emb={n_emb} is too large for FCI. Skipping.")

In [ ]:
# Example 5: Scan over basis sets
# Run Steps 0-2 for multiple basis sets and compare embedding quality

basis_results = {}

for basis in ["sto-3g", "def2-svp"]:
    run_dir = tempfile.mkdtemp(prefix=f"quenais_basis_{basis}_")

    cfg_b = Config(
        molecule    = "TiO2",
        basis       = basis,
        project_dir = run_dir,
    )
    cfg_b.validate().make_dirs().load_geometry()

    run_classical(cfg_b, force=True)
    run_asf(cfg_b, force=True)
    run_hamiltonian(cfg_b, force=True)

    with open(cfg_b.step2_file, "rb") as f:
        s2 = pickle.load(f)

    basis_results[basis] = {
        "n_emb"   : s2["n_emb"],
        "n_qubits": 2 * s2["n_emb"],
        "sv2_cov" : s2["sv2_cov"],
    }

print(f"{'Basis':12s}  {'n_emb':6s}  {'qubits':7s}  {'sv² cov':8s}")
print("-" * 40)
for basis, r in basis_results.items():
    print(f"{basis:12s}  {r['n_emb']:6d}  {r['n_qubits']:7d}  {r['sv2_cov']:8.4f}")

## 6. Visualizing Results

In [ ]:
# Generate all plots
run_viz(cfg, force=True, no_scan=False, no_quantum_scan=True)

# Display plots inline
from IPython.display import Image, display
import os

for plot in sorted(os.listdir(cfg.plots_dir)):
    if plot.endswith(".png"):
        print(f"\n--- {plot} ---")
        display(Image(os.path.join(cfg.plots_dir, plot)))

In [ ]:
# Load and display the summary CSV
import csv

csv_path = os.path.join(cfg.results_dir, "simulation_summary.csv")
with open(csv_path) as f:
    rows = list(csv.reader(f))

print(f"{'Parameter':<40} {'Value'}")
print("-" * 70)
for row in rows[1:]:   # skip header
    if row[1] != "N/A":
        print(f"{row[0]:<40} {row[1]}")

## 7. Running from the Command Line

Everything above can also be done from the terminal:

In [ ]:
# Show all CLI options
import subprocess
result = subprocess.run(["quenais-run", "--help"], capture_output=True, text=True)
print(result.stdout)

In [ ]:
# Examples of CLI usage (run in terminal, not here)
examples = """
# Full pipeline with defaults
quenais-run --molecule TiO2 --basis def2-svp

# Only classical methods
quenais-run --molecule TiO2 --steps 0

# Steps 0-2 only (no quantum)
quenais-run --molecule TiO2 --steps 0 1 2

# Different solver
quenais-run --molecule TiO2 --solver skqd --mapping bk

# SU2 ansatz instead of LUCJ
quenais-run --molecule TiO2 --ansatz su2 --shots 16384

# Skip geometry scan (faster)
quenais-run --molecule TiO2 --no-scan

# Force rerun ignoring cache
quenais-run --molecule TiO2 --force

# Different project directory
quenais-run --molecule TiO2 --project-dir /path/to/my/project
"""
print(examples)

## 8. Adding Your Own Molecule

1. Place your CIF file in `cif_files/YourMolecule.cif`
2. Change `molecule="YourMolecule"` in the Config
3. Check `charge` and `spin` match your system
4. Run the pipeline

In [ ]:
# Template for a new molecule
# Uncomment and fill in your values

# cfg_new = Config(
#     molecule    = "YourMolecule",   # must match YourMolecule.cif
#     basis       = "def2-svp",
#     charge      = 0,
#     spin        = 0,                # 0=singlet, 2=triplet, etc.
#     project_dir = PROJECT_DIR,
# )
# cfg_new.validate().make_dirs().load_geometry()
# print(f"Atoms: {cfg_new.atom_syms}")
# print(f"Geometry loaded: {cfg_new.n_atoms} atoms")

## 9. Understanding the Output Files

```
results/
├── step0_classical.pkl     Classical energies (HF, MP2, ...)
├── step1_asf.pkl           Active space: MO indices, tier, deviations
├── step2_hamiltonian.pkl   Embedding: h1e, h2e, bath SVs, ecore
├── step3_results.pkl       Quantum: energy, spin_sq, iterations
├── geometry_scan.pkl       PES scan data (if enabled)
├── simulation_summary.csv  All quantities in one table
└── plots/
    ├── plot1_energy_comparison.png   Bar chart of all methods
    ├── plot2_convergence.png         SQD iteration convergence
    ├── plot3_orbital_deviations.png  MP2 natural orbital deviations
    ├── plot4_bath_svs.png            Schmidt singular values
    ├── plot5_lowdin_heatmap.png      MO-to-atom Löwdin weights
    ├── plot6_geometry_scan.png       Potential energy surface
    └── plot7_summary.png             Full simulation dashboard
```